In [9]:
import einops 
from tqdm.notebook import tqdm

from torchsummary import summary

import torch
from torch import nn
import torchvision
import torch.optim as optim
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomCrop

In [10]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(device)

patch_size = 16
latent_size = 768
n_channels = 3
num_heads = 12
num_encoders = 12
dropout = 0.1
num_classes = 10
size = 224

epochs = 10
base_lr = 10e-3 
weight_decay = 0.03
batch_size = 8

cuda:0


In [11]:
class InputEmbedding(nn.Module):
    def __init__(self, patch_size=patch_size, n_channels=n_channels, device=device, latent_size=latent_size, batch_size=batch_size):
        super(InputEmbedding, self).__init__()
        self.latent_size = latent_size
        self.patch_size = patch_size
        self.n_channels = n_channels
        self.device = device
        self.batch_size = batch_size
        self.input_size = self.patch_size*self.patch_size*self.n_channels

        # Linear projection
        self.linearProjection = nn.Linear(self.input_size, self.latent_size)

        # Class token
        self.class_token = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)

        # Positional embedding
        self.pos_embedding = nn.Parameter(torch.randn(self.batch_size, 1, self.latent_size)).to(self.device)

    def forward(self, input_data):
        input_data = input_data.to(self.device)

        # Patchify input image
        patches = einops.rearrange(
            input_data, 'b c (h h1) (w w1) -> b (h w) (h1 w1 c)', h1=self.patch_size, w1=self.patch_size)

        #print(input_data.size())
        #print(patches.size())
        
        linear_projection = self.linearProjection(patches).to(self.device)
        b, n, _ = linear_projection.shape

        linear_projection = torch.cat((self.class_token, linear_projection), dim=1)
        pos_embed = einops.repeat(self.pos_embedding, 'b 1 d -> b m d', m=n+1)
    
        linear_projection += pos_embed

        return linear_projection


In [12]:
test_input = torch.randn((8, 3, 224, 224))
test_class = InputEmbedding().to(device)
embed_test = test_class(test_input)

In [13]:
class EncoderBlock(nn.Module):
    def __init__(self, latent_size=latent_size, num_heads=num_heads, device=device, dropout=dropout):
        super(EncoderBlock, self).__init__()

        self.latent_size = latent_size
        self.num_heads = num_heads
        self.device = device
        self.dropout = dropout

        # Normalization layer
        self.norm = nn.LayerNorm(self.latent_size)

        self.multihead = nn.MultiheadAttention(
            self.latent_size, self.num_heads, dropout=self.dropout
        )

        self.enc_MLP = nn.Sequential(
            nn.Linear(self.latent_size, self.latent_size*4),
            nn.GELU(),
            nn.Dropout(self.dropout),
            nn.Linear(self.latent_size*4, self.latent_size),
            nn.Dropout(self.dropout)
        )

    def forward(self, embedded_patches):
        firstnorm_out = self.norm(embedded_patches)
        attention_out = self.multihead(firstnorm_out, firstnorm_out, firstnorm_out)[0]

        # first residual connection
        first_added = attention_out + embedded_patches

        secondnorm_out = self.norm(first_added)
        ff_out = self.enc_MLP(secondnorm_out)

        return ff_out + first_added

In [14]:
test_encoder = EncoderBlock().to(device)
test_encoder(embed_test)

tensor([[[ 1.6270e+00,  3.5250e+00, -2.6266e-01,  ..., -1.8514e+00,
          -2.3007e-01,  2.1842e-02],
         [ 3.2333e-02,  2.5215e+00, -1.2891e+00,  ..., -1.7033e+00,
           2.0522e-02, -4.1129e-01],
         [-3.7077e-01,  1.5845e+00,  3.5275e-03,  ..., -1.6657e+00,
           1.1473e-01, -9.6388e-01],
         ...,
         [ 5.2352e-01,  1.4157e+00, -1.9970e+00,  ..., -1.0134e+00,
           5.0866e-01, -9.5202e-01],
         [-1.1053e+00,  1.2255e+00, -1.0423e-01,  ..., -1.6691e+00,
          -1.2540e-01,  2.4089e-01],
         [ 1.1056e+00,  1.3844e+00, -1.4294e+00,  ..., -1.8705e+00,
          -1.6621e-01, -6.4942e-01]],

        [[ 6.7435e-01, -1.5329e-01,  1.6589e+00,  ...,  3.4866e-01,
          -5.7379e-01,  2.0375e+00],
         [ 1.5931e+00, -5.2994e-01,  4.7188e-01,  ...,  5.6345e-01,
           7.0561e-01,  1.9114e+00],
         [ 1.4053e+00,  9.3735e-01,  1.4965e-02,  ...,  6.2721e-01,
          -7.5793e-01, -5.1026e-04],
         ...,
         [ 2.4331e+00,  1

In [15]:
class Vit(nn.Module):
    def __init__(self, num_encoders=num_encoders, latent_size=latent_size, device=device, num_classes=num_classes, dropout=dropout):
        super(Vit, self).__init__()
        self.num_encoder = num_encoders
        self.latent_size = latent_size
        self.device = device
        self.num_classes = num_classes
        self.dropout = dropout

        self.embedding = InputEmbedding()

        # Create the stack of encoders
        self.encStack = nn.ModuleList([EncoderBlock() for i in range(self.num_encoder)])

        self.MLP_head = nn.Sequential(
            nn.LayerNorm(self.latent_size),
            nn.Linear(self.latent_size, self.latent_size),
            nn.Linear(self.latent_size, self.num_classes)
        )

    def forward(self, test_input):
        enc_output = self.embedding(test_input)

        for enc_layer in self.encStack:
            enc_output = enc_layer(enc_output)

        cls_token_embed = enc_output[:, 0]

        return self.MLP_head(cls_token_embed)

In [16]:
model = Vit().to(device)
vit_output = model(test_input)
print(vit_output)
print(vit_output.size())

tensor([[-0.3194,  0.1707,  0.2516, -0.0822, -0.1771, -0.3371, -0.7288, -0.1366,
          0.6875,  0.7111],
        [-0.2485,  0.2612,  0.1054,  0.0108,  0.0124, -0.0882,  0.3797, -0.7561,
          0.2746,  0.6386],
        [ 0.0555,  0.2377,  0.0140,  0.2911, -0.4893, -0.1508, -0.2212,  0.4048,
          0.1060,  0.6389],
        [-0.0932,  0.0346,  0.0524, -0.0695, -0.3788, -0.0938,  0.1381, -0.0209,
         -0.0081,  0.1863],
        [ 0.4727,  0.2928, -0.1685, -0.1883, -0.0272, -0.1243, -0.1527, -0.2236,
          0.7449,  0.2315],
        [-0.3487,  0.4209,  0.1381, -0.0016, -0.4751, -0.1073, -0.0318, -0.0289,
          0.1906,  0.2809],
        [ 0.1410,  0.1959, -0.2818,  0.2811, -0.1629,  0.4749, -0.4793,  0.2936,
          0.3106,  0.2237],
        [-0.4160, -0.6378, -0.3535,  0.2376, -0.2677,  0.2268, -0.3278,  0.0293,
          0.4877, -0.1580]], device='cuda:0', grad_fn=<AddmmBackward0>)
torch.Size([8, 10])
